# reduce-op-mean-divide — ex2: harmonic mean across ranks via 1/x transform + all_reduce + divide + invert

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `reduce-op-mean-divide`. Running the final beacon cell reports progress against the `Distributed: reduce-op mean divide` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: reduce-op mean divide` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`reduce-op-mean-divide`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "reduce-op-mean-divide"
DD_SUBTOPIC = "Distributed: reduce-op mean divide"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Harmonic mean via sum-then-divide-then-invert — quick refresher

`dist.ReduceOp` still has no `MEAN`, no `HARMONIC_MEAN`, no `GEOMETRIC_MEAN`. Every named mean is built from the same recipe: transform → `all_reduce(SUM)` → divide → inverse transform.

For the **harmonic mean** of N rank-local values `x_r > 0`:
```
H = N / (sum_r 1/x_r)
```
Distributed implementation:
```python
tensor = t.tensor([1.0 / local_value], dtype=t.float32)    # transform
dist.all_reduce(tensor, op=dist.ReduceOp.SUM)              # sum 1/x_r
tensor /= world_size                                       # divide → mean of 1/x
harmonic = 1.0 / tensor.item()                             # inverse transform
```

**The in-place divide still matters.** Same `/=` vs `=` distinction as the arithmetic-mean case — keeps caller-held references stable.

**Why harmonic for rates / speeds.** Averaging samples/sec across ranks: arithmetic mean over-weights fast ranks (they finished more iterations). Harmonic mean weights by *time spent*, which is the throughput-correct aggregate.

### Exercise 2 — harmonic mean across ranks via 1/x transform + all_reduce + divide + invert

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the transform-sum-divide-invert recipe (1/x → `all_reduce(SUM)` → `/= world_size` → 1/.) to compute the harmonic mean of per-rank values on every rank.
> Keywords: harmonic-mean, all_reduce, transform-and-invert, throughput
> ```

**KCs targeted:** `transform-then-sum-then-divide-then-invert`, `throughput-aggregation`

Implement `ex2_harmonic_mean(rank, world_size, dist_module, local_value)`. Compute the harmonic mean `H = N / sum(1/x_r)` across ranks.

Steps inside the function:
1. Validate `local_value > 0` (harmonic mean is undefined on zero/negative inputs). Raise `ValueError` if not.
2. Build the reciprocal tensor: `tensor = t.tensor([1.0 / local_value], dtype=t.float32)`.
3. `dist_module.all_reduce(tensor, op=dist_module.ReduceOp.SUM)` — now `tensor[0]` is `sum_r 1/x_r`.
4. In-place divide: `tensor /= world_size` — now `tensor[0]` is the arithmetic mean of the reciprocals.
5. Invert: `harmonic = 1.0 / tensor.item()`.
6. Return `harmonic` — a Python float, identical on every rank.

**Use case.** Per-rank training throughput in samples/sec. Rank 0 might do 100 samples/sec, rank 1 might do 50; the ARITHMETIC mean (75) over-weights the fast rank. The HARMONIC mean = 2 / (1/100 + 1/50) = 66.67 — the time-weighted average, which is the throughput a downstream consumer actually sees.

Input: `rank`, `world_size` ints; `dist_module`; `local_value` positive float.
Output: float — harmonic mean, same on every rank.

In [ ]:
def ex2_harmonic_mean(rank: int, world_size: int, dist_module, local_value: float) -> float:
    if local_value <= 0:
        raise ValueError(f'harmonic mean undefined for non-positive input: {local_value}')
    tensor = t.tensor([1.0 / local_value], dtype=t.float32)
    dist_module.all_reduce(tensor, op=dist_module.ReduceOp.SUM)
    tensor /= world_size
    return 1.0 / tensor.item()


<details><summary>Solution</summary>

```python
def ex2_harmonic_mean(rank: int, world_size: int, dist_module, local_value: float) -> float:
    if local_value <= 0:
        raise ValueError(f'harmonic mean undefined for non-positive input: {local_value}')
    tensor = t.tensor([1.0 / local_value], dtype=t.float32)
    dist_module.all_reduce(tensor, op=dist_module.ReduceOp.SUM)
    tensor /= world_size
    return 1.0 / tensor.item()
```

**The 'transform-aggregate-invert' shape is general.** Geometric mean = log → SUM → /= N → exp. Quadratic mean = square → SUM → /= N → sqrt. All built from `all_reduce(SUM)` + in-place divide + element-wise nonlinearities.

**Why validate `> 0`.** `1.0 / 0.0` is `inf` in IEEE 754; `1.0 / -2.0` flips sign and silently produces a finite-but-wrong answer. Both are footguns the test explicitly checks against.

**ReduceOp.SUM works on the transformed values, not the originals.** This is the key insight ex1 hints at and ex2 makes concrete: the missing `ReduceOp.MEAN` is just sugar for SUM + divide. Any named mean is SUM + divide of the right transformed values.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()